# Avis de restaurants : benchmark reproductible sur Colab

Ce notebook exécute le code du dépôt : préparation des données, baselines,
CamemBERT et SFT LoRA de Qwen3-0.6B. Il ne contient aucun résultat prérempli.

**Commencer avec `SMOKE_TEST = True`.** Ce mode entraîne sur 32 avis, conserve
la validation complète et ne consulte pas le test. Après vérification, passer
à `False` pour les expériences complètes. Les GPU et la durée des sessions Colab
ne sont pas garantis. Les checkpoints sont conservés dans votre Google Drive.

Choisir **Exécution → Modifier le type d'exécution → GPU**, puis exécuter les
cellules dans l'ordre. L'accès Drive et l'envoi des TSV demandent votre intervention.


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

SMOKE_TEST = True
RUN_TEST_EVALUATION = False     # Activer après avoir figé les choix sur validation.
RUN_TAG = "recipe-v1"           # Nouveau nom si la recette change.
SEED = 42                       # Seed d'entraînement ; le split reste toujours à 42.
EXPERIMENT_NAME = "restaurant-sentiment-v1"
PROJECT_REF = "main"             # Résolu une seule fois puis épinglé dans Drive.
USE_4BIT = False                 # LoRA standard d'abord ; QLoRA optionnel.
RUN_CAMEMBERT = True
RUN_SFT = True
REPOSITORY = "https://github.com/Juven14s/restaurant-sentiment-analysis.git"

def run_command(args, cwd=None):
    print(" ".join(str(a) for a in args), flush=True)
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)


## 1. Monter Drive et épingler le code

Un nouveau `EXPERIMENT_NAME` crée un espace indépendant. Ne supprimez pas les
splits pour changer un réglage : conservez le protocole et créez un nouveau run.
Le code et les modèles sont épinglés à des commits précis pour les reprises.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT_HOME = Path("/content/drive/MyDrive") / EXPERIMENT_NAME
PROJECT_HOME.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/restaurant-sentiment-analysis")
lock_file = PROJECT_HOME / "code_revision.json"
if not REPO_DIR.exists():
    run_command(["git", "clone", REPOSITORY, REPO_DIR])
actual_origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
if actual_origin != REPOSITORY:
    raise RuntimeError("Ce dossier Colab contient un autre dépôt ; choisissez un nouveau dossier.")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("Des fichiers suivis ont été modifiés. Conservez vos changements avant de relancer la configuration.")
if lock_file.exists():
    revision = json.loads(lock_file.read_text())["commit"]
else:
    run_command(["git", "fetch", "origin", PROJECT_REF], cwd=REPO_DIR)
    revision = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD"], cwd=REPO_DIR, text=True).strip()
run_command(["git", "fetch", "origin", revision], cwd=REPO_DIR)
run_command(["git", "checkout", "--detach", revision], cwd=REPO_DIR)
if not lock_file.exists():
    lock_file.write_text(json.dumps({"repository": REPOSITORY, "commit": revision}, indent=2))
print("Code épinglé :", revision)


## 2. Installer les dépendances et vérifier le GPU

La recette conserve PyTorch fourni par Colab s'il est compatible et fixe les
versions de Transformers, TRL et PEFT. Si vous aviez déjà importé ces bibliothèques
avant l'installation, redémarrez la session puis reprenez les cellules dans l'ordre.


In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], cwd=REPO_DIR)
if USE_4BIT:
    run_command([sys.executable, "-m", "pip", "install", "-e", ".[quantization]"], cwd=REPO_DIR)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA : sélectionnez un runtime GPU et vérifiez sa disponibilité dans Colab.")
print("GPU :", torch.cuda.get_device_name(0))
print("Mémoire totale (GiB) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
run_command([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO_DIR)


## 3. Fournir les données et figer les splits

Envoyez `ftdataset_train.tsv` et `ftdataset_val.tsv` du cours. Ils sont copiés
dans votre Drive, jamais dans GitHub. Les colonnes requises sont Avis, Prix,
Cuisine, Service. Les anciennes partitions sont regroupées, dédupliquées puis
séparées en un nouveau protocole 80/10/10. Ce n'est pas un test externe inédit.

Les sources doivent rester identiques lors d'une reprise. Un changement de
source ou de protocole nécessite un nouvel espace d'expérience.


In [ ]:
RAW_DIR = PROJECT_HOME / "source_data"
RAW_DIR.mkdir(exist_ok=True)
expected = ("ftdataset_train.tsv", "ftdataset_val.tsv")
if any(not (RAW_DIR / name).exists() for name in expected):
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        if name not in expected:
            raise ValueError("Nom inattendu : utilisez exactement les deux noms TSV indiqués.")
        target = RAW_DIR / name
        if target.exists() and target.read_bytes() != content:
            raise ValueError("Une source différente est déjà conservée ; créez un nouvel EXPERIMENT_NAME.")
        target.write_bytes(content)
if any(not (RAW_DIR / name).exists() for name in expected):
    raise FileNotFoundError("Il faut fournir les deux fichiers TSV pour continuer.")

DATA_DIR = PROJECT_HOME / "data-text-seed42"
run_command([sys.executable, "-m", "restaurant_sentiment", "prepare",
    "--input", *(RAW_DIR / name for name in expected), "--output-dir", DATA_DIR,
    "--seed", "42", "--split-by", "text"], cwd=REPO_DIR)
data_manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print("Effectifs :", data_manifest["split_counts"])
print("Chevauchements :", data_manifest["overlap_checks"])


## 4. Épingler les modèles

L'inférence Qwen sans adaptation et LoRA utilisent le même checkpoint de départ.
Les révisions sont enregistrées une seule fois dans Drive. Aucun modèle n'est
publié et aucune clé privée n'est nécessaire pour ces modèles publics.


In [ ]:
from huggingface_hub import HfApi

model_lock = PROJECT_HOME / "model_revisions.json"
if model_lock.exists():
    MODEL_REVISIONS = json.loads(model_lock.read_text())
else:
    api = HfApi()
    MODEL_REVISIONS = {name: api.model_info(name).sha for name in (
        "almanach/camembert-base", "Qwen/Qwen3-0.6B")}
    model_lock.write_text(json.dumps(MODEL_REVISIONS, indent=2))
print(MODEL_REVISIONS)
RUNS_DIR = PROJECT_HOME / ("smoke_runs" if SMOKE_TEST else "full_runs")
RUNS_DIR.mkdir(exist_ok=True)


## 5. Entraîner et reprendre les runs

Les commandes s'exécutent dans des processus distincts pour libérer la mémoire
GPU entre modèles. Un run terminé est réutilisé. Un run interrompu reprend le
dernier checkpoint complet avec l'optimiseur et le scheduler. Le travail après
le dernier checkpoint est perdu. Si la mémoire est insuffisante pour CamemBERT,
réduisez son batch dans la recette et choisissez un nouveau nom de run ; pour
LoRA, testez QLoRA dans un espace distinct pour garder les comparaisons traçables.

Le mode smoke valide le fonctionnement. Ses pertes ne sont pas un benchmark.


In [ ]:
def training_recipe(backend):
    is_encoder = backend == "camembert"
    return dict(
        model_name="almanach/camembert-base" if is_encoder else "Qwen/Qwen3-0.6B",
        epochs=1.0 if SMOKE_TEST or not is_encoder else 3.0,
        learning_rate=3e-5 if is_encoder else 2e-4,
        batch_size=8 if is_encoder else 1,
        max_length=256 if is_encoder else 512,
        gradient_accumulation_steps=1 if is_encoder else 8,
    )

def train_run(backend):
    recipe = training_recipe(backend)
    out = RUNS_DIR / f"{backend}-{RUN_TAG}-seed{SEED}"
    manifest_path = out / "run_manifest.json"
    desired = {**recipe, "seed": SEED,
        "model_revision": MODEL_REVISIONS[recipe["model_name"]],
        "max_train_samples": 32 if SMOKE_TEST else None}
    if backend == "sft":
        desired.update(quantize_4bit=USE_4BIT, lora_rank=8)
    old = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    if old and any(old["config"].get(key) != value for key, value in desired.items()):
        raise ValueError("La recette a changé. Utilisez un nouveau répertoire de run.")
    if old and old["status"] == "completed" and (out / "final").is_dir():
        print("Run déjà terminé :", out)
        return out
    args = [sys.executable, "-m", "restaurant_sentiment", "train", "--backend", backend,
        "--data-dir", DATA_DIR, "--output-dir", out, "--seed", SEED, "--device", "cuda"]
    for key, value in desired.items():
        if key == "seed" or value is None or key == "quantize_4bit":
            continue
        args.extend(["--" + key.replace("_", "-"), value])
    if backend == "sft" and USE_4BIT:
        args.append("--quantize-4bit")
    checkpoints = sorted((p for p in out.glob("checkpoint-*")
        if all((p / name).exists() for name in
            ("trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth"))),
        key=lambda p: int(p.name.rsplit("-", 1)[1]))
    if checkpoints:
        args.extend(["--resume-from-checkpoint", checkpoints[-1]])
    elif old:
        raise RuntimeError("Run interrompu sans checkpoint complet. Préservez-le et choisissez un nouveau répertoire de run.")
    run_command(args, cwd=REPO_DIR)
    return out

CAMEMBERT_RUN = train_run("camembert") if RUN_CAMEMBERT else None


In [ ]:
SFT_RUN = train_run("sft") if RUN_SFT else None


## 6. Évaluation finale — désactivée en mode smoke

Avant cette étape, figez vos choix à partir de la validation, puis activez
`RUN_TEST_EVALUATION = True` avec `SMOKE_TEST = False`. Le test ne doit
pas servir à sélectionner prompts, hyperparamètres ou graines. Les prédictions
invalides restent comptées comme erreurs. Les résultats d'un seul run n'estiment
pas la variance entre graines. La séparation par restaurants serait une autre
expérience avec un autre test et un autre espace.


In [ ]:
METRIC_PATHS = []
if SMOKE_TEST or not RUN_TEST_EVALUATION:
    print("Évaluation test désactivée. Après validation du protocole : SMOKE_TEST=False et RUN_TEST_EVALUATION=True.")
else:
    for method in ("majority", "tfidf"):
        output = PROJECT_HOME / "baselines" / f"{method}-{RUN_TAG}-seed{SEED}"
        run_command([sys.executable, "-m", "restaurant_sentiment", "baseline",
            "--data-dir", DATA_DIR, "--method", method, "--output-dir", output, "--seed", SEED], cwd=REPO_DIR)
        METRIC_PATHS.append(output / "metrics.json")

    experiments = []
    if CAMEMBERT_RUN:
        experiments.append(("camembert", str(CAMEMBERT_RUN / "final"), CAMEMBERT_RUN, 256, 8))
    if RUN_SFT:
        experiments.append(("llm", "Qwen/Qwen3-0.6B", RUNS_DIR / f"qwen-prompt-{RUN_TAG}-seed{SEED}", 512, 1))
        experiments.append(("lora", str(SFT_RUN / "final"), SFT_RUN, 512, 1))
    for backend, model_path, output, max_length, batch_size in experiments:
        predictions = output / "predictions.jsonl"
        args = [sys.executable, "-m", "restaurant_sentiment", "predict", "--backend", backend,
            "--model-path", model_path, "--data-path", DATA_DIR / "test.jsonl",
            "--output-path", predictions, "--device", "cuda", "--batch-size", batch_size,
            "--max-length", max_length, "--max-new-tokens", "96"]
        if backend == "llm":
            args.extend(["--model-revision", MODEL_REVISIONS[model_path]])
        if USE_4BIT and backend in ("llm", "lora"):
            args.append("--quantize-4bit")
        run_command(args, cwd=REPO_DIR)
        metrics = output / "metrics.json"
        run_command([sys.executable, "-m", "restaurant_sentiment", "evaluate",
            "--references", DATA_DIR / "test.jsonl", "--predictions", predictions,
            "--output-path", metrics], cwd=REPO_DIR)
        METRIC_PATHS.append(metrics)


## 7. Lire les résultats et conserver les artefacts

Les métriques sont des fractions de 0 à 1. Le rapport contient le macro-F1 moyen
sur trois aspects, l'accuracy moyenne, l'exactitude jointe et le taux de conformité
JSON. Les détails par classe sont dans chaque `metrics.json`.

Pour plusieurs graines, répétez les runs complets avec `SEED = 43` puis `44`,
sans changer le split. Comparez les résultats seulement après avoir figé le
protocole. Ne publiez pas les TSV ou les prédictions brutes sans autorisation.


In [ ]:
if METRIC_PATHS:
    comparison_path = RUNS_DIR / f"comparison-{RUN_TAG}-seed{SEED}.json"
    run_command([sys.executable, "-m", "restaurant_sentiment", "summarize",
        "--metrics", *METRIC_PATHS, "--output-path", comparison_path], cwd=REPO_DIR)
    report = json.loads(comparison_path.read_text())
    for row in report["results"]:
        print(row["method"], "macro-F1 =", round(row["macro_f1"]["mean"], 4),
            "accuracy =", round(row["macro_accuracy"]["mean"], 4))
print("Artefacts conservés dans :", PROJECT_HOME)
